## RAG Movie Recommendation System

In [1]:
import pandas as pd
import kagglehub
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import NLTKTextSplitter
import chromadb



/Users/dani/Desktop/Codebase/AI-Agents-with-RAG-LLMs-and-Knowledge-Graphs/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.options.display.max_columns = None
pd.options.display.max_rows = None
pd.options.display.max_colwidth = None

In [3]:


# Download latest version
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset",output_dir="./Data")

print("Path to dataset files:", path)
#

Path to dataset files: ./Data


In [4]:
df=pd.read_csv(f"{path}/movies_metadata.csv", low_memory=False)

In [5]:
df.head(2)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg'}",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name': 'Family'}]",NaN,8844,tt0113497,en,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures.",17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'name': 'Teitler Film', 'id': 2550}, {'name': 'Interscope Communications', 'id': 10201}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [6]:
df = df[['original_title', 'overview']]

In [7]:
df.head(2)

,original_title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences."
1,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures."


In [8]:
import nltk
nltk.download('punkt_tab')

text_splitter = NLTKTextSplitter(chunk_size=1500, chunk_overlap=0)

def create_chunks(text):
    if pd.isna(text):
        return []
    return text_splitter.split_text(str(text))

df['chunks'] = df['overview'].apply(create_chunks)



[nltk_data] Downloading package punkt_tab to /Users/dani/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [9]:
df.head(2)

,original_title,overview,chunks
0,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.","[Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene.\n\nAfraid of losing his place in Andy's heart, Woody plots against Buzz.\n\nBut when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.]"
1,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures.","[When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room.\n\nAlan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures.]"


In [10]:
chunked_df= df.explode('chunks').reset_index(drop=True)

In [11]:
chunked_df.shape,df.shape

((45466, 3), (45466, 3))

In [12]:
chunked_df.head(2)

,original_title,overview,chunks
0,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.","Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene.\n\nAfraid of losing his place in Andy's heart, Woody plots against Buzz.\n\nBut when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences."
1,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures.","When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room.\n\nAlan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures."


In [13]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def create_embeddings(text):
    if not isinstance(text, str) or text.strip() == "":
        return None
    return embedder.encode(text).tolist()

chunked_df['embeddings'] = chunked_df['chunks'].apply(create_embeddings)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 24777.09it/s]


In [14]:
#drop na
chunked_df = chunked_df.dropna(subset=['embeddings']).reset_index(drop=True)

In [15]:
client=chromadb.Client()
collection=client.get_or_create_collection(name="movie_recommendation")

for idx, row in chunked_df.iterrows():
    collection.add(
        ids=[str(idx)],
        metadatas=[{"original_title": row['original_title'],
                    "chunk": row['chunks']}],
        embeddings=[row['embeddings']]
    )

In [16]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from chromadb import Client
from sentence_transformers import SentenceTransformer
import torch

In [17]:
sentence_model=SentenceTransformer('all-MiniLM-L6-v2')
model=AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1",
                                            device_map="auto")
tokenizer=AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

text_generator=pipeline(model=model, 
                        tokenizer=tokenizer, 
                        task="text-generation",
                        return_full_text=True,
                        max_new_tokens=800, 
                        device_map="auto")                                       

Loading weights: 100%|██████████| 291/291 [00:02<00:00, 108.69it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [19]:
def retrieve_relevant_chunks(query, collection, top_k=5):
    query_embedding = sentence_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    if not results['documents']:
        return [],[]

    chunks=[]
    titles=[]

    for metadata in results['metadatas'][0]:
        chunks.append(metadata['chunk'])
        titles.append(metadata['original_title'])
    
    return chunks,titles

def generate_recommendations(user_query, collection, text_generator):
    relevant_chunks, titles = retrieve_relevant_chunks(user_query, collection)

    if not relevant_chunks:
        return "Sorry, I couldn't find any relevant movies based on your query."

    context = "\n\n".join([f"Title: {title}\nOverview: {chunk}" for title, chunk in zip(titles, relevant_chunks)])

    prompt = f"""[INST] You are an expert movie recommender. 
       Instruction: You're an expert in movie suggestions. 
       Your task, should you choose to accept it, is to analyze carefully the context and come up with an 
       exhaustive answer to the following question:
       {user_query}

        \n\nYour Context:\n{context} 
        [/INST]"""
    response = text_generator(prompt)[0]['generated_text']
    return response

In [21]:
client = chromadb.Client()
collection = client.get_collection(name='movie_recommendation')

In [22]:
query = "What are some good movies to watch on Aliens or Space?"
top_k = 5

In [24]:
chunks, titles = retrieve_relevant_chunks(query, collection)
print(f"Retrieved Chunks: {chunks}")
print(f"Retrieved Titles: {titles}")

Retrieved Chunks: ['A farce satirizing extraterrestrial horror movies such as Alien.', "All over the world, people report they've been visited by aliens, taken aboard spaceships and medically examined.\n\nThe authorities appear to know all about these visits but won't acknowledge it publicly.\n\nThis film focuses on two 'victims' who struggle to live normal lives, but the aliens keep coming back.\n\nAll is explained.", 'Rocky Jones, Space Ranger fights space pirates over an invisible spaceship.', 'A feature film docu-comedy about UFOs, Aliens, Sightings, Abductions, Other Worldly Visitors, Extra Terrestrials, UFO Researchers, Government Cover-ups, Close Encounters, Flying Saucers, Alien Spacecraft, an Impact Site, Alien Bodies, an Alien Autopsy, the 509th Bomb Group, Atomic Bomb Testing, The Trinity Site, the White Sands Missile Test Range, Los Alamos, the crash of 1947 at Roswell, New Mexico, U.S.A. which became known as the Roswell Incident, and the people of the town of Roswell who 

In [26]:
answer=generate_recommendations(query, collection, text_generator)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [27]:
print(answer)

[INST] You are an expert movie recommender. 
       Instruction: You're an expert in movie suggestions. 
       Your task, should you choose to accept it, is to analyze carefully the context and come up with an 
       exhaustive answer to the following question:
       What are some good movies to watch on Aliens or Space?

        

Your Context:
Title: The Creature Wasn't Nice
Overview: A farce satirizing extraterrestrial horror movies such as Alien.

Title: Intruders
Overview: All over the world, people report they've been visited by aliens, taken aboard spaceships and medically examined.

The authorities appear to know all about these visits but won't acknowledge it publicly.

This film focuses on two 'victims' who struggle to live normal lives, but the aliens keep coming back.

All is explained.

Title: Manhunt in Space
Overview: Rocky Jones, Space Ranger fights space pirates over an invisible spaceship.

Title: Six Days in Roswell
Overview: A feature film docu-comedy about UFOs,